In [8]:
from calibration import load_calibration_sentences
from short_embedding import ShortEmbedding

### Phase 2 — Load Calibration Data

In [ ]:
from datasets import load_dataset

dataset = load_dataset("minhnguyent546/mc4-vi", split="validation", streaming=True)
sentences = []
for sample in dataset:
    text = sample["text"].strip()
    if text:
        sentences.append(text)
    if len(sentences) >= 2_000:
        break

print(f"Loaded {len(sentences)} calibration sentences")
print(sentences[:3])

RuntimeError: Broken pipe

In [14]:
# Extract sentences from JSON values (Vietnamese queries)
sentences = list(vi_json.values())[:10_000]
print(f"Loaded {len(sentences)} calibration sentences")
print(sentences[:3])

Loaded 10000 calibration sentences
['Đơn vị dự bị động viên là gì?', 'Quân nhân dự bị được xếp trong đơn vị dự bị động viên thì phải có trách nhiệm như thế nào?', 'Quân nhân dự bị được xếp trong đơn vị dự bị động viên có trách nhiệm gì?']


### Load Model

In [5]:
model = ShortEmbedding(
    model_name_or_path="jinaai/jina-embeddings-v5-text-nano-retrieval",
    n_prune_layers=2,
)
print(type(model._auto_model))
print(f"Layers: {len(model._get_layers())}")
model._get_layers()

Unrecognized keys in `rope_parameters` for 'rope_type'='default': {'factor'}


Loading weights:   0%|          | 0/110 [00:00<?, ?it/s]

Unrecognized keys in `rope_parameters` for 'rope_type'='default': {'factor'}
Unrecognized keys in `rope_parameters` for 'rope_type'='default': {'factor'}


<class 'transformers.models.eurobert.modeling_eurobert.EuroBertModel'>
Layers: 12


ModuleList(
  (0-11): 12 x EuroBertDecoderLayer(
    (self_attn): EuroBertAttention(
      (q_proj): Linear(in_features=768, out_features=768, bias=False)
      (k_proj): Linear(in_features=768, out_features=768, bias=False)
      (v_proj): Linear(in_features=768, out_features=768, bias=False)
      (o_proj): Linear(in_features=768, out_features=768, bias=False)
    )
    (mlp): EuroBertMLP(
      (gate_proj): Linear(in_features=768, out_features=3072, bias=False)
      (up_proj): Linear(in_features=768, out_features=3072, bias=False)
      (down_proj): Linear(in_features=3072, out_features=768, bias=False)
      (act_fn): SiLUActivation()
    )
    (input_layernorm): EuroBertRMSNorm((768,), eps=1e-05)
    (post_attention_layernorm): EuroBertRMSNorm((768,), eps=1e-05)
  )
)

### Phase 3 — Compute Block Influence (BI) per Layer

In [6]:
model.eval_importance(sentences, batch_size=32, angular=False)
print("Layer importances (cosine-based BI):")
for i, score in enumerate(model.importances):
    print(f"  Layer {i:2d}: {score:.4f}")

KeyboardInterrupt: 

In [ ]:
# Optional: angular distance metric (from Gromov et al.)
# Resets importances and recomputes using arccos similarity
model.importances = [0.0 for _ in model._get_layers()]
model.eval_importance(sentences, batch_size=32, angular=True)
print("Layer importances (angular BI):")
for i, score in enumerate(model.importances):
    print(f"  Layer {i:2d}: {score:.4f}")

### Phase 4 — Remove Least Important Layers

In [17]:
# Embed a few sentences before pruning for comparison
test_sentences = [
    "The cat sat on the mat.",
    "A quick brown fox jumps over the lazy dog.",
    "Machine learning is a subset of artificial intelligence.",
]
before_embs = model.get_embeddings(test_sentences)
print(f"Embeddings before pruning: {before_embs.shape}")

Embeddings before pruning: (3, 768)


In [18]:
removed = model.remove_layers()
print(f"Removed layers: {removed}")
print(f"Remaining layers: {len(model._get_layers())}")
model._get_layers()

Removed layers: [21, 19, 20]
Remaining layers: 21


ModuleList(
  (0-20): 21 x Gemma3DecoderLayer(
    (self_attn): Gemma3Attention(
      (q_proj): Linear(in_features=768, out_features=768, bias=False)
      (k_proj): Linear(in_features=768, out_features=256, bias=False)
      (v_proj): Linear(in_features=768, out_features=256, bias=False)
      (o_proj): Linear(in_features=768, out_features=768, bias=False)
      (q_norm): Gemma3RMSNorm((256,), eps=1e-06)
      (k_norm): Gemma3RMSNorm((256,), eps=1e-06)
    )
    (mlp): Gemma3MLP(
      (gate_proj): Linear(in_features=768, out_features=1152, bias=False)
      (up_proj): Linear(in_features=768, out_features=1152, bias=False)
      (down_proj): Linear(in_features=1152, out_features=768, bias=False)
      (act_fn): GELUTanh()
    )
    (input_layernorm): Gemma3RMSNorm((768,), eps=1e-06)
    (post_attention_layernorm): Gemma3RMSNorm((768,), eps=1e-06)
    (pre_feedforward_layernorm): Gemma3RMSNorm((768,), eps=1e-06)
    (post_feedforward_layernorm): Gemma3RMSNorm((768,), eps=1e-06)
  )
)

In [19]:
import torch

after_embs = model.get_embeddings(test_sentences)
print(f"Embeddings after pruning: {after_embs.shape}")

# Cosine similarity between before/after embeddings to measure drift
before_t = torch.tensor(before_embs)
after_t = torch.tensor(after_embs)
cosine_sim = torch.nn.functional.cosine_similarity(before_t, after_t)
for i, sim in enumerate(cosine_sim):
    print(f"  Sentence {i}: cosine similarity before vs after = {sim:.4f}")

Embeddings after pruning: (3, 768)
  Sentence 0: cosine similarity before vs after = 0.7059
  Sentence 1: cosine similarity before vs after = 0.7449
  Sentence 2: cosine similarity before vs after = 0.7379


### Explicit Layer Removal (optional)
Pass specific layer indices instead of relying on importance scores.

In [ ]:
# Example: remove specific layers by index
# model.remove_layers(layers_to_remove=[10, 11, 12])

In [ ]:
save_path = "../crosslingual_pruned"
model.st_model.save(save_path)
print(f"Pruned model saved to {save_path}")